## Avaliação Formal sobre o Conjunto de Teste

Para obter métricas rigorosas e comparáveis entre todos os modelos, foi executada uma
avaliação formal sobre o split de teste (613 imagens, 771 instâncias) para cada modelo
treinado nas Fases 1 e 2.

### Configuração da Avaliação

| Parâmetro | Valor | Justificação |
|-----------|-------|--------------|
| `split`   | `test` | Subconjunto estritamente isolado, nunca visto durante treino |
| `conf`    | `0.001` | Valor mínimo para varrer todos os thresholds e calcular mAP corretamente |
| `iou`     | `0.6` | Threshold de IoU para NMS durante a avaliação |
| `imgsz`   | `640` | Resolução de entrada consistente com o treino |

> **Nota:** O valor de `conf=0.001` não representa o threshold operacional recomendado.
> É utilizado exclusivamente para o cálculo do mAP, que requer a varredura de todos os
> níveis de confiança possíveis para construir as curvas Precision-Recall.

### Modelos Avaliados

| Modelo | Fase | Arquitetura |
|--------|------|-------------|
| `yolov8n_tatica_a`  | 1 | YOLOv8n — AdamW, lr0=0.01  |
| `yolov8n_tatica_b2` | 1 | YOLOv8n — AdamW, lr0=0.001 |
| `yolov8n_tatica_c`  | 1 | YOLOv8n — SGD,   lr0=0.001 |
| `yolov8s_fase2`     | 2 | YOLOv8s — SGD,   lr0=0.001 |
| `yolov8m_fase2`     | 2 | YOLOv8m — SGD,   lr0=0.001 |
| `yolov11s_fase2`    | 2 | YOLOv11s — SGD,  lr0=0.001 |
| `yoloV11m_fase2`    | 2 | YOLOv11m — SGD,  lr0=0.001 |

### Métricas Reportadas

Para cada modelo são reportadas as seguintes métricas sobre o conjunto de teste:

- **Globais:** `mAP@0.5`, `mAP@0.5:0.95`, `Precision`, `Recall`, `F1`
- **Por classe:** `P`, `R`, `F1`, `AP@0.5`, `AP@0.5:0.95` para cada uma das 6 classes

Os resultados são guardados automaticamente em cada pasta de modelo no Drive
(`runs/<nome_modelo>/test/`) e exportados em formato JSON para análise posterior.

In [ ]:
# Inferência sobre todos os modelos existentes com o dataset de TEST
# ATENÇÃO: confidence = 0.001 e IoU = 0.6 para poder calcular o mAP corretamente (varre todos os thresholds possíveis)

from ultralytics import YOLO
import shutil
from google.colab import files

DRIVE_BASE = "drive/MyDrive/Colab Notebooks/gloves_project/runs"
DATA_YAML  = "/content/Trabalho_IA-2/data.yaml"

MODELS = {
    "yolov8n_tatica_a" : f"{DRIVE_BASE}/yolov8n_tatica_a/weights/best.pt",
    "yolov8n_tatica_b2": f"{DRIVE_BASE}/yolov8n_tatica_b2/weights/best.pt",
    "yolov8n_tatica_c" : f"{DRIVE_BASE}/yolov8n_tatica_c/weights/best.pt",
    "yolov8s_fase2"    : f"{DRIVE_BASE}/yolov8s_fase2/weights/best.pt",
    "yolov8m_fase2"    : f"{DRIVE_BASE}/yolov8m_fase2/weights/best.pt",
    "yolov11s_fase2"   : f"{DRIVE_BASE}/yolov11s_fase2/weights/best.pt",
    "yoloV11m_fase2"   : f"{DRIVE_BASE}/yoloV11m_fase2/weights/best.pt",
}

for name, weights_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    model = YOLO(weights_path)

    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        conf=0.001,
        iou=0.6,
        plots=True,
        save_json=True,
        project=DRIVE_BASE,
        name=f"{name}/test",
        verbose=True,
    )

    print(f"\n--- {name} ---")
    print(f"mAP@0.5:      {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    print(f"Precision:    {metrics.box.mp:.4f}")
    print(f"Recall:       {metrics.box.mr:.4f}")
    f1_global = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-8)
    print(f"F1 (global):  {f1_global:.4f}")

    print("\nPor classe:")
    for i, class_name in model.names.items():
        p      = metrics.box.p[i]
        r      = metrics.box.r[i]
        f1     = 2 * p * r / (p + r + 1e-8)
        ap50   = metrics.box.ap50[i]
        ap5095 = metrics.box.ap[i]
        print(f"  {class_name:<25} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP@0.5={ap50:.3f}  AP@0.5:0.95={ap5095:.3f}")

# Zip e download de todos os resultados de teste
shutil.make_archive("/content/test_results", "zip", DRIVE_BASE)
files.download("/content/test_results.zip")

print("\nDownload started.")